In [1]:
pip install transformers torch

   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   --------------- ------------------------ 4.5/11.2 MB 25.1 MB/s eta 0:00:01
   -------------------------------------- - 10.7/11.2 MB 26.8 MB/s eta 0:00:01
   ---------------------------------------  11.0/11.2 MB 27.0 MB/s eta 0:00:01
   ---------------------------------------- 11.2/11.2 MB 15.9 MB/s  0:00:00
   ---------------------------------------- 0.0/719.8 kB ? eta -:--:--
   ---------------------------------------- 719.8/719.8 kB 13.5 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 20.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 26.9 MB/s  0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   -- ------------------------------------- 7.3/123.0 MB 34.8 MB/s eta 0:00:04
   ---- -------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.35.0 which is incompatible.


#### Next Word Prediction

In [88]:
## Step 1: Tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "The food was delicious.I was so happy to see"

tokens = tokenizer.tokenize(text)
print(tokens)

['The', 'Ġfood', 'Ġwas', 'Ġdelicious', '.', 'I', 'Ġwas', 'Ġso', 'Ġhappy', 'Ġto']


In [89]:
## Step 2: Convert Tokens to IDs
input_ids = tokenizer.encode(text)
print(input_ids)

[464, 2057, 373, 12625, 13, 40, 373, 523, 3772, 284]


In [90]:
### Step 3: Pass Through Embedding Layer
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained("gpt2")

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    embeddings = model.transformer.wte(inputs["input_ids"])

print(embeddings.shape)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

torch.Size([1, 10, 768])


In [91]:
### Step 4: Add Positional Encoding
## Final Embedding = Word Embedding + Position Embedding
position_embeddings = model.transformer.wpe(
    torch.arange(inputs["input_ids"].shape[1]).unsqueeze(0)
)

print(position_embeddings.shape)

torch.Size([1, 10, 768])


In [92]:
## Step 5: Self-Attention
## Step 6: Produce Logits
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(logits.shape)

torch.Size([1, 10, 50257])


For every token position, the model predicts probabilities over 50,257 vocabulary words.

In [93]:
## Step 7: Only Use the Last Position
next_token_logits = logits[:, -1, :]
next_token_logits

tensor([[-144.8661, -148.1142, -152.2806,  ..., -157.5947, -156.4989,
         -148.3836]])

In [94]:
### Step 8: Convert to Probabilities
probabilities = torch.softmax(next_token_logits, dim=-1)
probabilities

tensor([[2.6409e-05, 1.0260e-06, 1.5911e-08,  ..., 7.8307e-11, 2.3428e-10,
         7.8370e-07]])

In [95]:
### Step 9: Pick the Highest Probability
next_token_id = torch.argmax(probabilities, dim=-1)

next_word = tokenizer.decode(next_token_id)

print(next_word)

 see


In [1]:
from transformers import pipeline

In [4]:
pipeline("text")

KeyError: "Unknown task text, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [12]:
generator = pipeline("text-generation", model="gpt2")

result = generator(
    "The future of Machine Learning",
    num_return_sequences=2
)

print(result[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The future of Machine Learning

Machine Learning is a field that is in its infancy but is changing rapidly. The key to the future of machine learning is the use of machine learning algorithms to create useful tools that will help us understand and improve the human-computer interaction experience.

Machine Learning can be a daunting task but it will help us better understand the human-computer interaction experience. Understanding how to use machine learning techniques to teach machines how to work, to solve problems, to solve problems, and to learn tasks should be very easy for us. Machine learning algorithms can be used by many different industries to help us understand why people use them and learn new skills.

We are excited to announce the Open Source Open Machine Learning Project, which is currently working on a new approach to machine learning that is more easily understood and better understood than previous approaches. Open Machine Learning is a multi-platform project that can

In [8]:
?generator

Signature:      generator(text_inputs, **kwargs)
Type:           TextGenerationPipeline
String form:    TextGenerationPipeline: {'model': 'GPT2LMHeadModel', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text', 'output_modalities': ('text',)}
File:           c:\users\admin\anaconda3\lib\site-packages\transformers\pipelines\text_generation.py
Docstring:     
Language generation pipeline using any `ModelWithLMHead` or `ModelForCausalLM`. This pipeline predicts the words
that will follow a specified text prompt. When the underlying model is a conversational model, it can also accept
one or more chats, in which case the pipeline will operate in chat mode and will continue the chat(s) by adding
its response(s). Each chat takes the form of a list of dicts, where each dict contains "role" and "content" keys.

Unless the model you're using explicitly sets these generation parameters in its configuration files
(`generation_config.json`), the following default values will be used:
- m

### 1. Sentiment Analysis

In [15]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

print(classifier("The book had an interesting story-line"))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9996646642684937}]


### 2. Text Classification

In [16]:
classifier = pipeline("text-classification")

print(classifier("The product was pathetic. The quality was too bad."))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.999810516834259}]


### 3. Named Entity Recognition (NER)

In [21]:
from transformers import pipeline
ner = pipeline(
    "ner",
    aggregation_strategy="simple"
)
text = "Virat Kohli is an Indian Cricketer"
print(ner(text))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForTokenClassification LOAD REPORT from: gpt2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[{'entity_group': 'LABEL_1', 'score': np.float32(0.6720135), 'word': 'Vir', 'start': 0, 'end': 3}, {'entity_group': 'LABEL_0', 'score': np.float32(0.51048636), 'word': 'at', 'start': 3, 'end': 5}, {'entity_group': 'LABEL_1', 'score': np.float32(0.7444286), 'word': ' Kohli is an Indian Cricketer', 'start': 5, 'end': 34}]


### Zero-shot Classification

In [25]:
classifier = pipeline(
    "zero-shot-classification"
)

classifier(
    "I am satisfied with the services",
    candidate_labels=[
        "Complaint",
        "Praise",
        "Query"
    ]
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'I am satisfied with the services',
 'labels': ['Praise', 'Query', 'Complaint'],
 'scores': [0.7702361941337585, 0.17511723935604095, 0.05464654788374901]}

In [ ]:
from transformers import pipeline

# 1. Initialize the modern text-generation pipeline
generator = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507")

# 2. Format your question and context using standard Chat Templates
messages = [
    {
        "role": "user",
        "content": (
            "Context: Hugging Face v5 streamlined its architecture.\n"
            "Question: What did Hugging Face v5 do?"
        )
    }
]

# 3. Generate the response
output = generator(messages, max_new_tokens=100)
print(output[0]["generated_text"][-1]["content"])


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

C:\Users\Admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--Qwen--Qwen3-4B-Instruct-2507. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization")

long_text = """
The Transformer architecture was introduced in 2017 and completely revolutionized 
Natural Language Processing. Instead of processing text sequentially like older RNNs, 
it uses self-attention mechanisms to look at an entire sentence at once, making it 
massively parallelizable and highly accurate for complex language understanding tasks.
"""

result = summarizer(long_text, max_length=25, min_length=10)
print(result[0]['summary_text'])